In [2]:
# =====================================================================
# 03 - GELECEK TAHMİNİ İÇİN VERİ HAZIRLIĞI
#
# Bu script 04 ve 05 (tekil + ensemble) için ortak baseline kurar:
#   - model_giris.csv'yi yükler
#   - doktor = uzman + pratisyen + asistan birleşik kolonunu ekler
#   - 2025-2030 için il bazlı nüfus projeksiyonu üretir
#     (TÜİK projeksiyonu varsa onu kullanır, yoksa lineer ekstrapolasyon)
#
# Çıktılar:
#   - gelecek_baz.csv          (2002-2024 tam + 2025-2030 nüfus + diğer hedefler NaN)
#   - nufus_projeksiyonu.csv   (sadece 2025-2030 nüfus, kontrol amaçlı)
# =====================================================================

# %% [1] KÜTÜPHANELER & VERİ ========================================
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

df = pd.read_csv("model_giris.csv").sort_values(["il_adı", "yıl"]).reset_index(drop=True)
df["doktor"] = df["uzman"] + df["pratisyen"] + df["asistan"]
print("Veri:", df.shape)
print("İller:", df["il_adı"].nunique())
print("Yıl aralığı:", df["yıl"].min(), "→", df["yıl"].max())

HEDEFLER = ["doktor", "ebe", "diger", "hemsire", "yatak", "hastane"]

Veri: (1863, 12)
İller: 81
Yıl aralığı: 2002 → 2024


In [4]:
# %% [2] TÜİK PROJEKSİYONUNU OKU (varsa) ============================
TUIK_YOL = "hamveriler/tuik26-30hesaplanmis/Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx"

nufus_2025_2030 = {}    # {(il, yıl): nufus}

if os.path.exists(TUIK_YOL):
    try:
        tp = pd.read_excel(TUIK_YOL)
        print("\nTÜİK projeksiyonu okundu. Sütunlar:", list(tp.columns))
        # Sütun adları "il_adı" ve yıllar (2025, 2026, ...) varsayımı
        il_kolonu = None
        for c in tp.columns:
            if str(c).lower() in ("il_adı", "il", "il_ad", "il_adi"):
                il_kolonu = c; break
        if il_kolonu is None:
            il_kolonu = tp.columns[0]
        for _, r in tp.iterrows():
            il = str(r[il_kolonu]).strip()
            for y in range(2025, 2031):
                if y in tp.columns:
                    nufus_2025_2030[(il, y)] = float(r[y])
        print(f"TÜİK projeksiyonundan {len(nufus_2025_2030)} (il, yıl) çifti alındı.")
    except Exception as e:
        print(f"[UYARI] TÜİK dosyası okunamadı: {e}")


TÜİK projeksiyonu okundu. Sütunlar: ['TÜRKİYE İL BAZLI NÜFUS PROJEKSİYONU — TÜİK ANA SENARYO (GERÇEK VERİYLE GÜNCELLENMİŞ)', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
TÜİK projeksiyonundan 0 (il, yıl) çifti alındı.


In [6]:
# %% [3] EKSİK YILLAR İÇİN LİNEER EKSTRAPOLASYON ====================
# TÜİK'te bulamadıklarımız için son 10 yılın trendinden tahmin
eksik = []
for il in df["il_adı"].unique():
    seri = df[df["il_adı"] == il].dropna(subset=["nufus"]).tail(10)
    if len(seri) < 3:
        continue
    coef = np.polyfit(seri["yıl"], seri["nufus"], 1)
    for y in range(2025, 2031):
        if (il, y) not in nufus_2025_2030:
            nufus_2025_2030[(il, y)] = float(np.polyval(coef, y))
            eksik.append((il, y))

if eksik:
    print(f"\nLineer ekstrapolasyonla doldurulan: {len(eksik)} (il, yıl) çifti")


Lineer ekstrapolasyonla doldurulan: 486 (il, yıl) çifti


In [8]:
# %% [4] BASELINE DATAFRAME =========================================
# 2002-2024 mevcut + 2025-2030 boş satırlar (sadece nüfus dolu)
yeni_satirlar = []
for il in df["il_adı"].unique():
    for y in range(2025, 2031):
        yeni_satirlar.append({
            "il_adı": il, "yıl": y,
            "nufus": nufus_2025_2030.get((il, y), np.nan),
            **{h: np.nan for h in HEDEFLER}
        })

gelecek_df = pd.DataFrame(yeni_satirlar)
baz = pd.concat([df, gelecek_df], ignore_index=True).sort_values(["il_adı", "yıl"]).reset_index(drop=True)
print("\nBaseline boyut:", baz.shape)
print(baz[baz["yıl"].isin([2024, 2025, 2026])].head(8).to_string(index=False))

baz.to_csv("gelecek_baz.csv", index=False, encoding="utf-8-sig")
print("\n[OK] gelecek_baz.csv kaydedildi.")


Baseline boyut: (2349, 12)
        il_adı  yıl  hastane  yatak        nufus  asistan  diger    ebe  hemsire  pratisyen  uzman  doktor
         Adana 2024     32.0 7203.0 2.280484e+06   1502.0 7902.0 1722.0   6921.0     1384.0 2971.0  5857.0
         Adana 2025      NaN    NaN 2.300466e+06      NaN    NaN    NaN      NaN        NaN    NaN     NaN
         Adana 2026      NaN    NaN 2.311345e+06      NaN    NaN    NaN      NaN        NaN    NaN     NaN
      Adıyaman 2024     11.0 1077.0 6.110370e+05    125.0 2116.0  675.0   1929.0      439.0  403.0   967.0
      Adıyaman 2025      NaN    NaN 6.250166e+05      NaN    NaN    NaN      NaN        NaN    NaN     NaN
      Adıyaman 2026      NaN    NaN 6.260177e+05      NaN    NaN    NaN      NaN        NaN    NaN     NaN
Afyonkarahisar 2024     22.0 2380.0 7.501930e+05    440.0 2744.0  587.0   2381.0      496.0  713.0  1649.0
Afyonkarahisar 2025      NaN    NaN 7.608105e+05      NaN    NaN    NaN      NaN        NaN    NaN     NaN

[OK] gel

In [10]:
# %% [5] KONTROL DOSYASI ============================================
nf_df = pd.DataFrame([
    {"il_adı": il, **{y: nufus_2025_2030.get((il, y), np.nan) for y in range(2025, 2031)}}
    for il in sorted(df["il_adı"].unique())
])
nf_df.to_csv("nufus_projeksiyonu.csv", index=False, encoding="utf-8-sig")
print("[OK] nufus_projeksiyonu.csv kaydedildi (kontrol amaçlı).")
print(nf_df.head().to_string(index=False))

[OK] nufus_projeksiyonu.csv kaydedildi (kontrol amaçlı).
        il_adı         2025         2026         2027         2028         2029         2030
         Adana 2.300466e+06 2.311345e+06 2.322223e+06 2.333101e+06 2.343980e+06 2.354858e+06
      Adıyaman 6.250166e+05 6.260177e+05 6.270189e+05 6.280200e+05 6.290212e+05 6.300223e+05
Afyonkarahisar 7.608105e+05 7.659676e+05 7.711247e+05 7.762818e+05 7.814389e+05 7.865960e+05
       Aksaray 4.503925e+05 4.563322e+05 4.622719e+05 4.682116e+05 4.741512e+05 4.800909e+05
        Amasya 3.447114e+05 3.465732e+05 3.484351e+05 3.502969e+05 3.521587e+05 3.540206e+05
